# LINE WORKS API: AiNote ノートの取得 (一覧・検索・詳細)

LINE WORKS AiNote (AI 議事録) のノートを API で取得するサンプルです。

- 認証は **User Account 認証** のアクセストークンが必要です (Service Account 認証は使えません)
- スコープは `ainote.read` を付けてください
- トークンの取得方法は Qiita の記事「LINE WORKS API トークン取得 (User Account 認証)」を参照してください
- トークンは実行時に入力します (ノートブックには保存されません)。Colab の「シークレット」に `LINEWORKS_ACCESS_TOKEN` を登録しておくと入力を省略できます

実行順: 1 → 2 (一覧) → 3 (検索) → 4 (詳細)。2 か 3 で得た `noteId` を 4 で使います。

In [ ]:
# @title 1. 設定 (実行するとアクセストークンの入力を求められます)
import requests
import json
from getpass import getpass

# トークンはノートブックに保存しない。
# Colab の「シークレット」に LINEWORKS_ACCESS_TOKEN を登録してあればそれを使い、なければ入力欄で聞く。
ACCESS_TOKEN = ""
try:
    from google.colab import userdata
    ACCESS_TOKEN = userdata.get("LINEWORKS_ACCESS_TOKEN")
except Exception:
    pass
if not ACCESS_TOKEN:
    ACCESS_TOKEN = getpass("アクセストークン: ").strip()

BASE_URL = "https://www.worksapis.com/v1.0"
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

def hms(milliseconds):
    """ミリ秒を H:MM:SS に整形する。audioDuration と発言の offset はミリ秒。"""
    total = int(milliseconds or 0) // 1000
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"

def call(path, params=None):
    """API を呼び、エラー時は API が返す code / description を表示する。"""
    res = requests.get(f"{BASE_URL}/{path}", headers=HEADERS, params=params, timeout=30)
    if res.status_code >= 400:
        print(f"HTTP {res.status_code}")
        print(res.text)
        return None
    return res.json()

print("設定完了" if ACCESS_TOKEN else "アクセストークンが入力されていません。")

In [ ]:
# @title 2. ノート一覧の取得 (要約も文字起こしも含まれない)
COUNT = 10  # @param {type:"integer"}

data = call("users/me/ainote/notes", {"count": COUNT})
if data:
    for n in data["notes"]:
        print(f"{n['createdTime'][:16]}  {hms(n['audioDuration']):>7}  {n['title']}")
        print(f"    noteId: {n['noteId']}")
    print("\nnextCursor:", data.get("responseMetaData", {}).get("nextCursor"))

In [ ]:
# @title 3. ノートの検索 (query は必須。呼び出し上限は 60 回/分)
QUERY = "営業"  # @param {type:"string"}

data = call("users/me/ainote/search", {"query": QUERY, "count": 10})
if data:
    print(f"{len(data['notes'])} 件")
    for n in data["notes"]:
        print(f"{n['createdTime'][:16]}  {hms(n['audioDuration']):>7}  {n['title']}")
        print(f"    noteId: {n['noteId']}")

In [ ]:
# @title 4. ノート詳細の取得 (要約と文字起こし)
NOTE_ID = ""  # @param {type:"string"}
MAX_BLOCKS = 10  # @param {type:"integer"}

note = call(f"users/me/ainote/notes/{NOTE_ID}") if NOTE_ID else print("NOTE_ID を入力してください。")
if note:
    print(f"タイトル : {note['title']}")
    print(f"長さ     : {hms(note['audioDuration'])}")
    print(f"attendees: {[a['attendeeName'] for a in note.get('attendees') or []]}")

    # 要約は作成者が AiNote 上でテンプレートを選んで生成したときだけ入る
    summaries = note.get("summaries") or []
    print(f"\n■ 要約 ({len(summaries)} 件)")
    for s in summaries:
        print(f"--- {s['summaryName']} ({s['summaryType']}) ---")
        print(s["content"])
    if not summaries:
        print("(要約は生成されていません)")

    scripts = note.get("scripts") or []
    print(f"\n■ 文字起こし (全 {len(scripts)} ブロック、先頭 {min(MAX_BLOCKS, len(scripts))} 件)")
    for b in scripts[:MAX_BLOCKS]:
        text = b["text"].strip().replace("\n", " ")  # ブロック内の改行を潰して 1 行にする
        print(f"[{hms(b['startOffset'])}] {b.get('attendeeName') or '話者不明'}: {text}")

    # attendees は LINE WORKS ユーザーとして紐づいた参加者だけ。話者の一覧は scripts から集める
    speakers = sorted({b.get("attendeeName") or "話者不明" for b in scripts})
    print(f"\n■ 話者 (scripts から集計): {speakers}")

In [ ]:
# @title (参考) 生のレスポンスを見る
if NOTE_ID:
    raw = call(f"users/me/ainote/notes/{NOTE_ID}")
    if raw:
        raw["scripts"] = raw.get("scripts", [])[:2]  # 長いので先頭 2 ブロックだけ
        print(json.dumps(raw, ensure_ascii=False, indent=2))